# Anime Colab Studio — acceso sin códigos de correo

Selecciona el modelo, ComfyUI o AUTOMATIC1111, Cloudflare o ngrok y LoRAs. Los túneles apuntan a un proxy que solicita una contraseña una vez por sesión; las peticiones de generación y WebSocket quedan cubiertas por la misma cookie del navegador.

**Inicio:** habilita una GPU en Colab, ejecuta las dos primeras celdas y selecciona las preferencias. Puedes guardar resultados PNG y checkpoints en Google Drive. Si no defines `WEBUI_PASSWORD` en los Secretos de Colab, el notebook genera una contraseña de sesión y la muestra al finalizar. No se envían códigos de correo.

**Diagnóstico:** la celda 3 consulta el estado real directamente, sin depender de los widgets del panel. La celda 4 es un inicio alternativo con salida continua.


In [ ]:
# @title 1. Instalar dependencias mínimas
%pip -q install -U ipywidgets huggingface_hub requests 'aiohttp>=3.10,<4'


In [ ]:
# @title 2. Abrir panel interactivo con progreso y logs recuperables
from google.colab import output
output.enable_custom_widget_manager()
from pathlib import Path
import importlib
import subprocess
import sys
project = Path("/content/ai_nsfw_source")
if not (project / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Mairon31/ai_nsfw.git",
                    str(project)], check=True)
else:
    subprocess.run(["git", "-C", str(project), "fetch", "--depth", "1",
                    "origin", "master"], check=True)
    subprocess.run(["git", "-C", str(project), "reset", "--hard",
                    "origin/master"], check=True)
sys.path.insert(0, str(project))
importlib.invalidate_caches()
for name in list(sys.modules):
    if name == "studio" or name.startswith("studio."):
        del sys.modules[name]
from studio.dashboard import make_dashboard
make_dashboard()


In [ ]:
# @title 3. Diagnóstico (ejecuta esta celda si el panel no avanza o quieres ver el progreso)
# Puedes repetir esta celda cuantas veces quieras mientras el instalador trabaja.
import json
from pathlib import Path
from studio import runtime
state = runtime.snapshot()
print("ESTADO:", state.get("stage", "sin iniciar"))
print("MENSAJE:", state.get("message", "sin datos"))
print("AVANCE ESTIMADO:", state.get("percent", 0), "%")
print("COMPROBACIÓN EXTERNA:", state.get("tunnel_verified", "pendiente"))
if state.get("access_password"):
    print("CONTRASEÑA DE SESIÓN:", state["access_password"])
elif state.get("password_source") == "WEBUI_PASSWORD":
    print("CONTRASEÑA: usa el secreto WEBUI_PASSWORD de Colab")
if state.get("url"):
    print("URL REMOTA:", state["url"])
print("PROCESOS:", json.dumps(runtime.read_json(runtime.PIDS), ensure_ascii=False))
print("ACTIVO:", runtime.process_active())
print("ÚLTIMOS LOGS (instalador + UI + túnel):")
print(runtime.recent_log(65))


### Inicio directo (alternativa)
Si el botón del panel se queda en *Preparando*, guarda preferencias, activa `INICIAR_EN_ESTA_CELDA` y ejecuta la siguiente celda. Verás el instalador en la consola activa. **No hace falta usar este modo si el botón funciona.**


In [ ]:
# @title 4. Opcional: iniciar en esta celda y ver cada log en vivo
# Activa esta casilla SOLO si al usar el botón "Instalar / iniciar" se atasca.
INICIAR_EN_ESTA_CELDA = False # @param {type:"boolean"}
if INICIAR_EN_ESTA_CELDA:
    from studio import launcher, runtime
    print("Modo directo de Colab: preferencias cargadas desde Google Drive.")
    cfg = launcher.load_settings()
    secrets = {k: launcher.secret(k) for k in
               ("HF_TOKEN", "NGROK_AUTHTOKEN", "CIVITAI_API_TOKEN", "WEBUI_PASSWORD")}
    codigo = runtime.launch_foreground(cfg, secrets)
    estado = runtime.snapshot()
    print("\n--- ESTADO FINAL ---")
    print(estado.get("stage"), ":", estado.get("message"))
    if estado.get("url"):
        print("ABRIR INTERFAZ:", estado["url"])
else:
    print("Modo directo desactivado; utiliza el panel de la celda 2.")
